#### Compliance Engine & Risk Scoring Logic

In [1]:
import pandas as pd
import numpy as np

#### 1. Load data

In [2]:
df_scenarios = pd.read_csv("warehouse_scenarios.csv")
df_checklist = pd.read_csv("compliance_checklist.csv")

print("Scenarios:", df_scenarios.shape)
print(df_scenarios[["scenario_id", "country", "title"]])

print("\nChecklist:", df_checklist.shape)
print(df_checklist[["check_id", "category", "requirement", "weight"]])

Scenarios: (5, 5)
  scenario_id      country                                    title
0     SCN-001  South Sudan          Flood-damaged warehouse in Juba
1     SCN-002        Yemen       Access-blocked warehouse in Sana'a
2     SCN-003     Ethiopia  High-turnover nutrition store in Tigray
3     SCN-004  Afghanistan        Cold-chain medical store in Kabul
4     SCN-005   Bangladesh              Rohingya camp NFI warehouse

Checklist: (10, 6)
  check_id                  category  \
0  CHK-001  Warehouse Infrastructure   
1  CHK-002  Warehouse Infrastructure   
2  CHK-003        Stock Organization   
3  CHK-004        Stock Organization   
4  CHK-005             Documentation   
5  CHK-006             Documentation   
6  CHK-007         Expiry Management   
7  CHK-008         Expiry Management   
8  CHK-009          Safety & Hygiene   
9  CHK-010          Safety & Hygiene   

                                         requirement  weight  
0  Warehouse is secure (locked doors, fencing, li.

#### 2. Define Scoring Rules

Possible user answers for each checklist item:
- Yes      → 100% points
- Partial  → 50% points
- No       → 0% points
- N/A      → excluded from total (weight moved to denominator)

Total possible points = sum of weights of answered items
Compliance % = (earned points / total possible) × 100

In [3]:
def calculate_compliance_score(answers_dict):
    """
    answers_dict: {check_id: answer} where answer is 'Yes', 'Partial', 'No', 'N/A'
    Returns: dict with score, earned, possible, percentage
    """
    earned = 0.0
    possible = 0.0
    
    for _, row in df_checklist.iterrows():
        chk_id = row["check_id"]
        weight = row["weight"]
        
        if chk_id not in answers_dict:
            continue  # skip unanswered
            
        answer = answers_dict[chk_id].strip().title()
        
        if answer == "N/A":
            continue  # exclude from calculation
            
        possible += weight
        
        if answer == "Yes":
            earned += weight
        elif answer == "Partial":
            earned += weight * 0.5
        # No = +0
    
    if possible == 0:
        percentage = 0.0
    else:
        percentage = round((earned / possible) * 100, 1)
    
    return {
        "compliance_percentage": percentage,
        "points_earned": earned,
        "points_possible": possible,
        "answered_items": len(answers_dict)
    }

#### 3. Risk Category & Severity Mapping

Based on compliance % and number of critical fails

In [4]:
def get_risk_category(compliance_pct, critical_fails=0):
    """
    critical_fails = number of failed items with weight ≥ 20
    """
    if compliance_pct >= 90:
        level = "Low Risk"
        color = "green"
    elif compliance_pct >= 70:
        level = "Medium Risk"
        color = "orange"
    elif compliance_pct >= 50:
        level = "High Risk"
        color = "red"
    else:
        level = "Critical Risk"
        color = "darkred"
    
    if critical_fails >= 2:
        level = f"{level} + Critical Items"
    
    return level, color

#### 4. Generate Findings & Corrective Actions

Rule-based suggestions (can be expanded later with LLM or more rules)

In [5]:
def generate_findings_and_actions(answers_dict):
    findings = []
    
    for _, row in df_checklist.iterrows():
        chk_id = row["check_id"]
        if chk_id not in answers_dict:
            continue
            
        answer = answers_dict[chk_id].strip().title()
        if answer in ["No", "Partial"]:
            finding = {
                "check_id": chk_id,
                "requirement": row["requirement"],
                "answer": answer,
                "weight": row["weight"],
                "severity": "High" if row["weight"] >= 18 else "Medium",
                "suggested_action": f"Immediately address {row['requirement'].lower()}. "
                                   f"{'Conduct full stock count.' if 'stock' in row['requirement'].lower() else ''}"
                                   f"{'Move to quarantine area and document loss.' if 'expired' in row['requirement'].lower() else ''}"
                                   f"{'Install/repair pest control measures.' if 'pest' in row['requirement'].lower() else ''}"
                                   f"Review and update SOPs."
            }
            findings.append(finding)
    
    # Sort by severity & weight
    findings_sorted = sorted(findings, key=lambda x: (-x["weight"], x["severity"] == "Medium"))
    
    return findings_sorted

#### 5. Test the Engine (Example)

In [6]:
# Example user answers for one scenario
example_answers = {
    "CHK-001": "Yes",        # secure warehouse
    "CHK-002": "Partial",    # ventilation partial
    "CHK-003": "No",         # FIFO not implemented
    "CHK-004": "Yes",
    "CHK-005": "No",         # ledger not updated
    "CHK-006": "Yes",
    "CHK-007": "Partial",
    "CHK-008": "No",         # expired items in main area
    "CHK-009": "N/A",
    "CHK-010": "Yes"
}

# Run scoring
score_result = calculate_compliance_score(example_answers)
print("Compliance Result:")
print(score_result)

# Risk category
critical_fails = sum(1 for v in example_answers.values() if v.title() == "No" and df_checklist[df_checklist["check_id"] == list(example_answers.keys())[list(example_answers.values()).index(v)]]["weight"].iloc[0] >= 20)
risk_level, risk_color = get_risk_category(score_result["compliance_percentage"], critical_fails)
print(f"\nRisk Level: {risk_level} ({risk_color})")

# Findings
findings = generate_findings_and_actions(example_answers)
print("\nFindings & Suggested Actions:")
for f in findings[:5]:  # show top 5
    print(f"• {f['requirement']} → {f['answer']} (weight {f['weight']})")
    print(f"  Action: {f['suggested_action']}\n")

Compliance Result:
{'compliance_percentage': 52.2, 'points_earned': 82.0, 'points_possible': 157.0, 'answered_items': 10}

Risk Level: High Risk (red)

Findings & Suggested Actions:
• No expired items in main storage area → No (weight 25)
  Action: Immediately address no expired items in main storage area. Move to quarantine area and document loss.Review and update SOPs.

• Expiry dates tracked and visible → Partial (weight 22)
  Action: Immediately address expiry dates tracked and visible. Review and update SOPs.

• FIFO/FEFO system visibly implemented → No (weight 18)
  Action: Immediately address fifo/fefo system visibly implemented. Review and update SOPs.

• Updated stock ledger / bin cards for each item → No (weight 15)
  Action: Immediately address updated stock ledger / bin cards for each item. Conduct full stock count.Review and update SOPs.

• Adequate ventilation and protection from weather → Partial (weight 12)
  Action: Immediately address adequate ventilation and protecti